## ライブラリの読み込み

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

## データ読み込み

In [2]:
ROOT = Path.cwd()

if ROOT.name == "smartphone_addiction":
    ROOT = ROOT.parent.parent

DATA_DIR = ROOT / "data" / "smartphone_addiction" / "raw"
OUTPUT_DIR = ROOT / "output" / "smartphone_addiction"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train.shape)
print("test :", test.shape)

train: (691369, 14)
test : (296302, 13)


## 元データの作成

In [3]:
TARGET = "addicted_label"

X_base = train.drop(
    columns=[TARGET, "id"]
).copy()

y = train[TARGET].copy()

X_test_base = test.drop(
    columns=["id"]
).copy()

## 特徴量作成関数

In [4]:
def add_behavior_features(df):
    df = df.copy()

    eps = 1e-6

    # 休日と通常時のスクリーン時間差
    df["weekend_screen_gap"] = (
        df["weekend_screen_time"]
        - df["daily_screen_time_hours"]
    )

    # SNS利用割合
    df["social_media_ratio"] = (
        df["social_media_hours"]
        / (df["daily_screen_time_hours"] + eps)
    )

    # ゲーム利用割合
    df["gaming_ratio"] = (
        df["gaming_hours"]
        / (df["daily_screen_time_hours"] + eps)
    )

    # 1回のアプリ起動あたり通知数
    df["notifications_per_open"] = (
        df["notifications_per_day"]
        / (df["app_opens_per_day"] + eps)
    )

    # スクリーン1時間あたりアプリ起動回数
    df["opens_per_screen_hour"] = (
        df["app_opens_per_day"]
        / (df["daily_screen_time_hours"] + eps)
    )

    # スクリーン1時間あたり通知数
    df["notifications_per_screen_hour"] = (
        df["notifications_per_day"]
        / (df["daily_screen_time_hours"] + eps)
    )

    return df

## 実際に特徴量を追加

In [5]:
X_behavior = add_behavior_features(X_base)
X_test_behavior = add_behavior_features(X_test_base)

print("Before:", X_base.shape)
print("After :", X_behavior.shape)

Before: (691369, 12)
After : (691369, 18)


In [6]:
new_features = [
    "weekend_screen_gap",
    "social_media_ratio",
    "gaming_ratio",
    "notifications_per_open",
    "opens_per_screen_hour",
    "notifications_per_screen_hour"
]

X_behavior[new_features].head()

,weekend_screen_gap,social_media_ratio,gaming_ratio,notifications_per_open,opens_per_screen_hour,notifications_per_screen_hour
0,NaN,NaN,NaN,3.210526,NaN,NaN
1,NaN,0.180904,NaN,4.000000,3.182579,12.730316
2,2.38,NaN,NaN,2.233333,11.787817,26.326124
3,2.24,0.196262,0.221184,1.191489,14.641742,17.445480
4,2.19,0.166964,0.250893,NaN,NaN,NaN


## inf が発生していないか確認

In [7]:
np.isinf(
    X_behavior.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

## CatBoost 用のカテゴリ列処理

In [8]:
def prepare_for_catboost(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    categorical_cols = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    for col in categorical_cols:
        X_train[col] = (
            X_train[col]
            .fillna("Missing")
            .astype(str)
        )

        X_test[col] = (
            X_test[col]
            .fillna("Missing")
            .astype(str)
        )

    return X_train, X_test, categorical_cols

## CV用関数を作成

In [9]:
def evaluate_catboost(X, y, categorical_cols):
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []
    best_iterations = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):
        print(f"\n===== Fold {fold} =====")

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(
            iterations=6500,
            learning_rate=0.05,
            depth=7,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=500,
            allow_writing_files=False
        )

        model.fit(
            X_train,
            y_train,
            cat_features=categorical_cols,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=200
        )

        valid_pred = model.predict_proba(
            X_valid
        )[:, 1]

        score = roc_auc_score(
            y_valid,
            valid_pred
        )

        scores.append(score)
        best_iterations.append(
            model.get_best_iteration()
        )

        print(
            f"Fold {fold} AUC: {score:.6f}"
        )

    scores = np.array(scores)

    print("\n各Fold:")
    print(scores)

    print("\nMean AUC:", scores.mean())
    print("Std AUC :", scores.std())
    print("Best iterations:", best_iterations)

    return (
        scores.mean(),
        scores.std(),
        best_iterations
    )

## Behavior Features を評価

In [11]:
# 準備
X_behavior_cb, X_test_behavior_cb, categorical_cols = (
    prepare_for_catboost(
        X_behavior,
        X_test_behavior
    )
)

C:\Users\Owner\AppData\Local\Temp\ipykernel_28428\2286189730.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [12]:
# cvの実行
behavior_mean_auc, behavior_std_auc, behavior_best_iterations = (
    evaluate_catboost(
        X_behavior_cb,
        y,
        categorical_cols
    )
)


===== Fold 1 =====
0:	test: 0.9088930	best: 0.9088930 (0)	total: 343ms	remaining: 37m 12s
500:	test: 0.9531237	best: 0.9531237 (500)	total: 1m 43s	remaining: 20m 37s
1000:	test: 0.9583233	best: 0.9583233 (1000)	total: 3m 20s	remaining: 18m 20s
1500:	test: 0.9601960	best: 0.9601960 (1499)	total: 4m 57s	remaining: 16m 31s
2000:	test: 0.9612580	best: 0.9612580 (2000)	total: 6m 36s	remaining: 14m 51s
2500:	test: 0.9618217	best: 0.9618222 (2498)	total: 8m 14s	remaining: 13m 11s
3000:	test: 0.9621961	best: 0.9621963 (2994)	total: 9m 53s	remaining: 11m 32s
3500:	test: 0.9623789	best: 0.9623789 (3500)	total: 11m 33s	remaining: 9m 54s
4000:	test: 0.9624944	best: 0.9624955 (3998)	total: 13m 14s	remaining: 8m 16s
4500:	test: 0.9625684	best: 0.9625709 (4495)	total: 14m 53s	remaining: 6m 36s
5000:	test: 0.9626210	best: 0.9626210 (5000)	total: 16m 35s	remaining: 4m 58s
5500:	test: 0.9626457	best: 0.9626485 (5367)	total: 18m 15s	remaining: 3m 18s
6000:	test: 0.9626652	best: 0.9626669 (5972)	total: 1

## 結果表

In [13]:
results = [
    {
        "experiment": "CatBoost baseline",
        "mean_auc": 0.963516,
        "std_auc": 0.000343
    },
    {
        "experiment": "CatBoost + behavior features",
        "mean_auc": behavior_mean_auc,
        "std_auc": behavior_std_auc
    }
]

results_df = pd.DataFrame(results)

results_df.sort_values(
    "mean_auc",
    ascending=False
)

,experiment,mean_auc,std_auc
0,CatBoost baseline,0.963516,0.000343
1,CatBoost + behavior features,0.963291,0.000422


## １特徴量ずつ試す

In [14]:
feature_functions = {
    "weekend_screen_gap": lambda df:
        df["weekend_screen_time"]
        - df["daily_screen_time_hours"],

    "social_media_ratio": lambda df:
        df["social_media_hours"]
        / (df["daily_screen_time_hours"] + 1e-6),

    "gaming_ratio": lambda df:
        df["gaming_hours"]
        / (df["daily_screen_time_hours"] + 1e-6),

    "notifications_per_open": lambda df:
        df["notifications_per_day"]
        / (df["app_opens_per_day"] + 1e-6),

    "opens_per_screen_hour": lambda df:
        df["app_opens_per_day"]
        / (df["daily_screen_time_hours"] + 1e-6),

    "notifications_per_screen_hour": lambda df:
        df["notifications_per_day"]
        / (df["daily_screen_time_hours"] + 1e-6)
}

## 欠損数の情報を考慮に入れる

In [15]:
# 「この人は全部で何項目欠損しているのか」という情報を入れる
def add_missing_features(df):
    df = df.copy()

    df["missing_count"] = (
        df.isna().sum(axis=1)
    )

    return df

In [16]:
# behavior features に追加
X_behavior_missing = add_missing_features(
    X_behavior
)

X_test_behavior_missing = add_missing_features(
    X_test_behavior
)

In [17]:
# 準備
X_bm_cb, X_test_bm_cb, categorical_cols_bm = (
    prepare_for_catboost(
        X_behavior_missing,
        X_test_behavior_missing
    )
)

C:\Users\Owner\AppData\Local\Temp\ipykernel_28428\2286189730.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [18]:
# cvを実行
bm_mean_auc, bm_std_auc, bm_best_iterations = (
    evaluate_catboost(
        X_bm_cb,
        y,
        categorical_cols_bm
    )
)


===== Fold 1 =====
0:	test: 0.9108621	best: 0.9108621 (0)	total: 200ms	remaining: 21m 41s
500:	test: 0.9532650	best: 0.9532650 (500)	total: 1m 42s	remaining: 20m 22s
1000:	test: 0.9582093	best: 0.9582093 (1000)	total: 3m 22s	remaining: 18m 33s
1500:	test: 0.9601364	best: 0.9601372 (1499)	total: 5m 4s	remaining: 16m 53s
2000:	test: 0.9611069	best: 0.9611069 (2000)	total: 6m 45s	remaining: 15m 12s
2500:	test: 0.9617029	best: 0.9617029 (2500)	total: 8m 25s	remaining: 13m 29s
3000:	test: 0.9620251	best: 0.9620251 (3000)	total: 10m 6s	remaining: 11m 47s
3500:	test: 0.9622748	best: 0.9622748 (3500)	total: 11m 47s	remaining: 10m 5s
4000:	test: 0.9623981	best: 0.9623981 (4000)	total: 13m 28s	remaining: 8m 24s
4500:	test: 0.9625165	best: 0.9625181 (4491)	total: 15m 8s	remaining: 6m 43s
5000:	test: 0.9625947	best: 0.9625964 (4986)	total: 16m 49s	remaining: 5m 2s
5500:	test: 0.9626201	best: 0.9626212 (5490)	total: 18m 31s	remaining: 3m 21s
6000:	test: 0.9626685	best: 0.9626693 (5991)	total: 20m 

In [19]:
# 結果表を追加
results.append({
    "experiment":
        "CatBoost + behavior + missing count",
    "mean_auc":
        bm_mean_auc,
    "std_auc":
        bm_std_auc
})

pd.DataFrame(results).sort_values(
    "mean_auc",
    ascending=False
)

,experiment,mean_auc,std_auc
0,CatBoost baseline,0.963516,0.000343
1,CatBoost + behavior features,0.963291,0.000422
2,CatBoost + behavior + missing count,0.963249,0.000405


## 最終モデル

In [20]:
X_final = add_behavior_features(X_base)
X_final = add_missing_features(X_final)

X_test_final = add_behavior_features(
    X_test_base
)
X_test_final = add_missing_features(
    X_test_final
)

X_final, X_test_final, categorical_cols_final = (
    prepare_for_catboost(
        X_final,
        X_test_final
    )
)

C:\Users\Owner\AppData\Local\Temp\ipykernel_28428\2286189730.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [21]:
# cv の best iteration を活用
final_iterations = int(
    np.mean(bm_best_iterations)
)

print(final_iterations)

5826


## 全trainデータで学習

In [22]:
final_model = CatBoostClassifier(
    iterations=final_iterations,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=500,
    allow_writing_files=False
)

final_model.fit(
    X_final,
    y,
    cat_features=categorical_cols_final
)

0:	total: 229ms	remaining: 22m 13s
500:	total: 2m	remaining: 21m 20s
1000:	total: 4m	remaining: 19m 20s
1500:	total: 6m	remaining: 17m 19s
2000:	total: 8m 2s	remaining: 15m 22s
2500:	total: 10m 6s	remaining: 13m 25s
3000:	total: 12m 9s	remaining: 11m 26s
3500:	total: 14m 14s	remaining: 9m 27s
4000:	total: 16m 19s	remaining: 7m 26s
4500:	total: 18m 23s	remaining: 5m 24s
5000:	total: 20m 27s	remaining: 3m 22s
5500:	total: 22m 31s	remaining: 1m 19s
5825:	total: 23m 52s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=5826, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=500)

## 提出用ファイル作成

In [23]:
test_pred = final_model.predict_proba(
    X_test_final
)[:, 1]

In [24]:
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

submission = sample_submission.copy()

submission[TARGET] = test_pred

In [25]:
# 確認
assert submission["id"].equals(test["id"])
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

display(submission.head())

,id,addicted_label
0,691369,0.999455
1,691370,0.945762
2,691371,0.928818
3,691372,0.992336
4,691373,0.996824


In [26]:
# 保存
submission_path = (
    OUTPUT_DIR
    / "submission_catboost_feature_engineering.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(submission_path)

c:\Users\Owner\Documents\projects\kaggle-study\output\smartphone_addiction\submission_catboost_feature_engineering.csv
